# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET http_keep_alive=true;")
con.execute(f"CREATE SECRET hf_sec (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

table_name = "fact_content_daily_performance"

query = f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/{table_name}/**/*.parquet'
WHERE month = '2026-03'
LIMIT 500
"""

df_sample = con.execute(query).df()
df_sample.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [18]:
df_sample.isna().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
client_has_gsc,0
client_has_ga4,0
gsc_data_available,0
ga4_data_available,500
gsc_impressions,0
gsc_clicks,0
gsc_sum_position,0


## 1. Unit of analysis + time window

*## Part 1: Data Contract

### 1) Row Grain
One row in this slice represents the daily aggregated search performance (clicks, impressions, position, CTR) for a single unique URL belonging to a specific client on a single date.

### 2) Table Selection
`fact_content_daily_performance`

### 3) Time Window
Mid-panel month: **March 2026 (`2026-03`)**.

### 4) Prediction Target (Label / Proxy)
`clicks` — Predicting the number of organic search clicks a URL will generate on a given target date.

### 5) Deliberate Exclusion
Records with missing/null URLs, zero-impression test entries, or incomplete daily logging windows are deliberately excluded to prevent noise and non-stationary signals in downstream modeling.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET http_keep_alive=true;")
con.execute(f"CREATE SECRET hf_sec (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}');")

# المسار الصحيح للملفات
dataset_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# -------------------------------------------------------------
# Query 1: Prove Grain (Should return 0 rows if unique)
# -------------------------------------------------------------
print("--- 1. Grain Verification ---")
q1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) as row_count
FROM '{dataset_path}'
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
"""
df_grain = con.execute(q1).df()
print("Duplicate Rows Count:", len(df_grain))
display(df_grain)

# -------------------------------------------------------------
# Query 2: Row Count & Date Span
# -------------------------------------------------------------
print("\n--- 2. Row Count & Date Span ---")
q2 = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM '{dataset_path}'
WHERE month = '2026-03';
"""
df_span = con.execute(q2).df()
display(df_span)

# -------------------------------------------------------------
# Query 3: Availability Verification
# -------------------------------------------------------------
print("\n--- 3. Availability Check (gsc_data_available IS TRUE) ---")
q3 = f"""
SELECT
    COUNT(*) AS available_rows
FROM '{dataset_path}'
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
"""
df_avail = con.execute(q3).df()
display(df_avail)

--- 1. Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate Rows Count: 0


,client_hash_id,content_hash_id,report_date,row_count



--- 2. Row Count & Date Span ---


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- 3. Availability Check (gsc_data_available IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 2. Fields: feature / label / context / excluded

*### 2. Fields Breakdown

#### 🔹 Label
* **`gsc_clicks`**: The target variable representing organic search traffic generated on the prediction date.

#### 🔹 Features
* **`feat_avg_impressions_7d`**: 7-day historical rolling average of `gsc_impressions` prior to the prediction date.
* **`feat_avg_position_7d`**: 7-day historical rolling average of `gsc_avg_position` prior to the prediction date.
* **`feat_sum_clicks_7d`**: 7-day historical rolling sum of `gsc_clicks` prior to the prediction date.
* **`feat_avg_ga4_sessions_7d`**: 7-day historical rolling average of `ga4_sessions` prior to the prediction date.
* **`day_of_week`**: Categorical temporal feature extracted directly from `report_date`.

#### 🔹 Context
* **`report_date`**: Temporal identifier used for time-series windowing and historical feature calculation.
* **`client_hash_id`**: Identifier hash for the client entity.
* **`content_hash_id`**: Identifier hash for the specific URL/content page.
* **`month`**: Partition key for dataset filtering (`2026-03`).

#### 🔹 Excluded & Reasons
* **`ga4_pageviews` / `ga4_users` / `ga4_engaged_sessions`**: **Excluded** due to high multicollinearity with `ga4_sessions` to prevent redundant feature impact.
* **`ai_chatgpt` / `ai_perplexity` / `ai_claude` / `ai_copilot` / `ai_gemini` / `ai_meta` / `ai_other`**: **Excluded** due to extreme data sparsity (mostly zero values in this specific slice), which introduces non-informative noise.
* **`gsc_sum_position`**: **Excluded** because `gsc_avg_position` provides a normalized and accurate representation of ranking without scaling bias.
* **`client_has_gsc` / `client_has_ga4`**: **Excluded** as they act as single-value constant flags within the filtered `gsc_data_available IS TRUE` slice.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define field buckets explicitly for downstream modeling
LABEL_COL = "gsc_clicks"

CONTEXT_COLS = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "month"
]

FEATURE_COLS = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks" # Will be transformed into historical rolling features in Part 3
]

EXCLUDED_COLS = [
    "gsc_sum_position",
    "client_has_gsc", "client_has_ga4",
    "gsc_data_available", "ga4_data_available",
    "ga4_pageviews", "ga4_sessions", "ga4_users",
    "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral",
    "sessions_social", "sessions_paid", "sessions_ai",
    "ai_chatgpt", "ai_perplexity", "ai_gemini",
    "ai_copilot", "ai_claude", "ai_meta", "ai_other",
    "scroll_events"
]

print(f" Label Column selected: '{LABEL_COL}'")
print(f" Context Columns ({len(CONTEXT_COLS)}): {CONTEXT_COLS}")
print(f" Base Feature Inputs ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f" Total Excluded Columns: {len(EXCLUDED_COLS)}")

 Label Column selected: 'gsc_clicks'
 Context Columns (4): ['report_date', 'client_hash_id', 'content_hash_id', 'month']
 Base Feature Inputs (3): ['gsc_impressions', 'gsc_avg_position', 'gsc_clicks']
 Total Excluded Columns: 24


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

# 1. Connect and initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET http_keep_alive=true;")
con.execute(f"CREATE SECRET hf_sec (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}');")

dataset_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# ==============================================================================
# Claim 1: Grain Verification (client_hash_id + content_hash_id + report_date)
# Expectation: 0 duplicate rows
# ==============================================================================
print(" [Claim 1] Verifying Row Grain Uniqueness...")
q_grain = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) as duplicate_count
FROM '{dataset_path}'
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
"""
df_grain = con.execute(q_grain).df()
print(f"-> Duplicates found: {len(df_grain)} (Should be 0)")
display(df_grain)

# ==============================================================================
# Claim 2: Total Counts & Date Window Bounds (March 2026)
# Expectation: Verify total slice size, start date, and end date
# ==============================================================================
print("\n [Claim 2] Verifying Row Counts & Time Window...")
q_window = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) as unique_clients,
    COUNT(DISTINCT content_hash_id) as unique_urls
FROM '{dataset_path}'
WHERE month = '2026-03';
"""
df_window = con.execute(q_window).df()
display(df_window)

# ==============================================================================
# Claim 3: Availability & Missing Values Check
# Expectation: Verify active GSC rows vs GA4 sparsity/missingness
# ==============================================================================
print("\n [Claim 3] Verifying Availability & Missing Values (GSC vs GA4)...")
q_missing = f"""
SELECT
    COUNT(*) AS total_slice_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_rows,
    COUNT(gsc_clicks) AS non_null_gsc_clicks,
    COUNT(gsc_impressions) AS non_null_gsc_impressions,
    COUNT(ga4_sessions) AS non_null_ga4_sessions,
    COUNT(ai_chatgpt) AS non_null_ai_chatgpt
FROM '{dataset_path}'
WHERE month = '2026-03';
"""
df_missing = con.execute(q_missing).df()
display(df_missing)

🔍 [Claim 1] Verifying Row Grain Uniqueness...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-> Duplicates found: 0 (Should be 0)


,client_hash_id,content_hash_id,report_date,duplicate_count



📊 [Claim 2] Verifying Row Counts & Time Window...


,total_rows,min_report_date,max_report_date,unique_clients,unique_urls
0,9841378,2026-03-01,2026-03-31,55,331437



⚠️ [Claim 3] Verifying Availability & Missing Values (GSC vs GA4)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_slice_rows,gsc_available_rows,non_null_gsc_clicks,non_null_gsc_impressions,non_null_ga4_sessions,non_null_ai_chatgpt
0,9841378,3611061,9841378,9841378,6822637,6822637


## 4. Data limits

*## 4. Data Limits

This dataset provides daily organic search performance metrics, but it possesses fundamental limitations that constrain model inference and interpretation:

### 1) GSC-Only Historical Rows (Missing Behavioral Signals)
* **The Limitation:** The slice relies primarily on Google Search Console (GSC) data, while Google Analytics 4 (GA4) metrics (`ga4_sessions`, `pageviews`, engagement time) are entirely missing/null in this partition (`0` valid records).
* **Impact:** The data can tell us **how visible** a URL was on Google (`impressions`, `clicks`, `position`), but it **can NEVER tell us what users did after landing on the page** (e.g., bounce rate, conversion, read duration, or actual user retention).

### 2) Unbalanced History & Cold-Start URLs
* **The Limitation:** New URLs published mid-month or clients onboarded recently lack a full 7-day or 14-day historical window prior to prediction dates.
* **Impact:** The model cannot accurately evaluate performance for newly published content without introducing artificial zero-padding or truncation bias.

### 3) Window Overlaps & Autocorrelation
* **The Limitation:** Rolling window aggregation features (e.g., 7-day average clicks) share heavily overlapping data points across consecutive days for the same URL.
* **Impact:** Highly dynamic intra-week spikes (e.g., sudden viral trends or Google algorithm updates mid-week) will be smoothed out, masking immediate short-term fluctuations.

### 4) Unobserved External Factors
* **The Limitation:** Search performance is heavily driven by external, untracked dynamics such as seasonal demand, Google core algorithm updates, competitor ranking shifts, and offline marketing campaigns.
* **Impact:** The dataset contains no features for competitor activity or algorithm updates, meaning sudden performance shifts cannot be causally explained by the model alone.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==============================================================================
# Part 3: Build 5 Historical Features using SQL Window Functions
# Note: Using 'ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING' avoids Data Leakage!
# ==============================================================================

q_features = f"""
WITH sorted_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_clicks AS label_target_clicks,
        gsc_impressions,
        gsc_avg_position
    FROM '{dataset_path}'
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
)
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    label_target_clicks,

    -- Feature 1: Rolling 7-day Average of Impressions
    AVG(gsc_impressions) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS feat_avg_impressions_7d,

    -- Feature 2: Rolling 7-day Average of Search Ranking Position
    AVG(gsc_avg_position) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS feat_avg_position_7d,

    -- Feature 3: Rolling 7-day Sum of Past Clicks
    SUM(label_target_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS feat_sum_clicks_7d,

    -- Feature 4: Rolling 3-day Average of Clicks
    AVG(label_target_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
    ) AS feat_avg_clicks_3d,

    -- Feature 5: Day of Week (1 = Sunday, 7 = Saturday)
    DAYOFWEEK(CAST(report_date AS DATE)) AS feat_day_of_week

FROM sorted_data
ORDER BY client_hash_id, content_hash_id, report_date;
"""

df_features = con.execute(q_features).df()
print(f" Generated DataFrame with shape: {df_features.shape}")
display(df_features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Generated DataFrame with shape: (3611061, 9)


,client_hash_id,content_hash_id,report_date,label_target_clicks,feat_avg_impressions_7d,feat_avg_position_7d,feat_sum_clicks_7d,feat_avg_clicks_3d,feat_day_of_week
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,2026-03-26,0,NaN,NaN,NaN,NaN,4
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-01,0,NaN,NaN,NaN,NaN,0
2,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-02,0,6.000000,6.166667,0.0,0.000000,1
3,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-03,0,7.500000,9.527778,0.0,0.000000,2
4,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-04,0,6.666667,15.018519,0.0,0.000000,3
5,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-05,1,7.000000,14.420139,0.0,0.000000,4
6,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-06,0,7.000000,16.078968,1.0,0.333333,5
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-07,0,7.833333,15.454696,1.0,0.333333,6
8,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-08,0,7.428571,14.732596,1.0,0.333333,0
9,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-09,0,8.285714,15.292120,1.0,0.000000,1


In [26]:
# ==============================================================================
# Part 4: Data Leakage Demonstration
# Creating a leaked feature from the PREDICTION DAY itself (SAME DAY)
# ==============================================================================

# 1. Clean q_features string by stripping trailing semicolons
clean_q_features = q_features.strip().rstrip(';')

# 2. Build the Leakage Query
q_leakage = f"""
WITH base_features AS (
    {clean_q_features}
)
SELECT
    f.*,
    -- LEAKED FEATURE: Including same-day impressions (Future knowledge)
    raw.gsc_impressions AS leaked_same_day_impressions
FROM base_features f
JOIN '{dataset_path}' raw
  ON f.client_hash_id = raw.client_hash_id
 AND f.content_hash_id = raw.content_hash_id
 AND f.report_date = raw.report_date
WHERE raw.month = '2026-03';
"""

# 3. Execute query
df_leakage = con.execute(q_leakage).df()

print(" Data Leakage Feature Demonstration:")
print("Notice how 'leaked_same_day_impressions' holds exact same-day search visibility:\n")
display(df_leakage[['report_date', 'label_target_clicks', 'feat_avg_impressions_7d', 'leaked_same_day_impressions']].head(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

⚠️ Data Leakage Feature Demonstration:
Notice how 'leaked_same_day_impressions' holds exact same-day search visibility:



,report_date,label_target_clicks,feat_avg_impressions_7d,leaked_same_day_impressions
0,2026-03-01,0,NaN,20
1,2026-03-01,0,NaN,1
2,2026-03-01,1,NaN,125
3,2026-03-01,0,NaN,7
4,2026-03-01,0,NaN,11


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.